# synthetic-ambient-fhir-25 — dataset exploration

25 synthetic clinical encounters (one per patient). Each record pairs an ambient
conversation **transcript** with the resulting clinical **note**, an **after-visit
summary**, the patient's **chart background**, and structured **FHIR R4** context.

Shared helpers live in `explore.py`; run `python explore.py` for the full text report.

In [ ]:
import json
from collections import Counter

import pandas as pd

from explore import load_records, parse_date, age_at

records = load_records()
len(records)

## Encounter overview as a DataFrame

One row per encounter with the fields you'll most often filter on.

In [ ]:
rows = []
for r in records:
    m = r["metadata"]
    p = r["patient_context"]["patient"]
    rows.append({
        "date": parse_date(m["date"]),
        "visit_title": m["visit_title"],
        "visit_type": m["visit_type"],
        "gender": p["gender"],
        "age": age_at(p["birthDate"], m["date"]),
        "transcript_words": len(r["transcript"].split()),
        "note_words": len(r["note"].split()),
        "avs_words": len(r["after_visit_summary"].split()),
        "fhir_at_visit": sum(m["related_resource_counts"].values()),
        "chart_resources": sum(r["patient_context"]["longitudinal_summary"]["resource_counts"].values()),
    })

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
df

In [ ]:
# Quick numeric + categorical summaries
display(df.describe())
display(df["visit_type"].value_counts())
display(df["gender"].value_counts())

In [ ]:
# Age distribution and text sizes at a glance
df.plot.scatter(x="transcript_words", y="note_words", title="transcript vs note length");

## FHIR Observations — parse the coded values

Observations are the richest resource type. Each carries a coded label and either a
`valueQuantity` (numeric + unit), a `valueCodeableConcept` (coded answer), a
`valueString`, or `component`s (e.g. blood-pressure systolic/diastolic). This flattens them.

In [ ]:
def obs_value(o):
    """Return a human-readable value for one FHIR Observation."""
    if "valueQuantity" in o:
        q = o["valueQuantity"]
        return f"{q.get('value')} {q.get('unit', '')}".strip()
    if "valueCodeableConcept" in o:
        return o["valueCodeableConcept"].get("text")
    if "valueString" in o:
        return o["valueString"]
    if "component" in o:
        parts = []
        for c in o["component"]:
            q = c.get("valueQuantity", {})
            parts.append(f"{c['code'].get('text', '?')}={q.get('value')} {q.get('unit', '')}".strip())
        return "; ".join(parts)
    return None


obs_rows = []
for r in records:
    for o in r["encounter_fhir"]["related_resources"].get("Observation", []):
        obs_rows.append({
            "encounter": r["metadata"]["visit_title"],
            "observation": o["code"].get("text"),
            "value": obs_value(o),
        })

obs = pd.DataFrame(obs_rows)
print(len(obs), "observations across all encounters")
obs["observation"].value_counts().head(20)

In [ ]:
# Example: pull every recorded Body Mass Index value
obs[obs["observation"].str.contains("Body mass index", case=False, na=False)]

## Transcript speaker-turn analysis

Transcripts are speaker-labeled (`DR:`, `PT:`, `NURSE:`, `FAMILY:`). This counts
turns and words per speaker — useful for measuring who dominates the conversation.

In [ ]:
import re

TURN = re.compile(r"^([A-Z]+):\s*(.*)$")


def speaker_turns(transcript):
    """Yield (speaker, text) for each labeled line in a transcript."""
    speaker, buf = None, []
    for line in transcript.splitlines():
        m = TURN.match(line.strip())
        if m:
            if speaker:
                yield speaker, " ".join(buf)
            speaker, buf = m.group(1), [m.group(2)]
        elif speaker:
            buf.append(line.strip())
    if speaker:
        yield speaker, " ".join(buf)


turns, words = Counter(), Counter()
for r in records:
    for spk, text in speaker_turns(r["transcript"]):
        turns[spk] += 1
        words[spk] += len(text.split())

spk_df = pd.DataFrame({"turns": turns, "words": words}).fillna(0).astype(int)
spk_df["words_per_turn"] = (spk_df["words"] / spk_df["turns"]).round(1)
spk_df.sort_values("words", ascending=False)

## Inspect one full encounter

Change the index to read the transcript → note → after-visit-summary pipeline for any record.

In [ ]:
i = 0
r = records[i]
print(r["metadata"]["visit_title"], "—", parse_date(r["metadata"]["date"]))
print("\n=== TRANSCRIPT ===\n", r["transcript"][:1500])
print("\n=== NOTE ===\n", r["note"][:1500])
print("\n=== AFTER-VISIT SUMMARY ===\n", r["after_visit_summary"])